# Gdańsk Regional Championship: Day 2 team analysis

This notebook explores the 110 teams above the **Day 2 end** divider in the Gdańsk Regional Championship standings. It compares team compositions, groups teams connected by an exact similarity score, and demonstrates a damage matchup.

Keep `gdansk_day_2.json` beside this notebook and run the notebook with this folder as the working directory. The `pyvgc` package must be installed in the active Python environment.

In [ ]:
import json
from pyvgc import (
    find_team_pairs_by_score,
    group_identical_teamplayers,
    pairwise_score_matrix,
)

with open("gdansk_day_2.json", encoding="utf-8") as data_file:
    standings_data = json.load(data_file)

teams = {
    entry["rank"]: {"player": entry["player"], "team": entry["team"]}
    for entry in standings_data[0:29]
}
print(f"Loaded {len(teams)} teams ({sum(len(t['team']) for t in teams.values())} Pokémon sets).")
print("Standings source: https://standings.limitlessvgc.com/0044/standings")

## Inspect the standings teams

Each record includes the player name and six sets containing Pokémon, item, ability, stat alignment (represented as nature by the package), and four moves.

In [ ]:
for rank, entry in teams.items():
    roster = ", ".join(pokemon["name"] for pokemon in entry["team"])
    print(f"{rank:>3}. {entry['player']}: {roster}")

## Compare team similarity

The score starts at 102 for two identical six-Pokémon teams.

More documentation about this function in the future.

A **higher score means more similar teams**. This implementation accepts the raw website team dictionaries, so it can also analyze sets that are outside the package's current M-C legality snapshot.

In [ ]:
score_matrix = pairwise_score_matrix(teams)
column_width = max(4, len(str(max(teams))) + 1)

print("Rows and columns are standings ranks; identical teams score 102.")
print(" " * column_width + "".join(f"{rank:>{column_width}}" for rank in teams))
for rank, scores in score_matrix.items():
    print(f"{rank:>{column_width}}" + "".join(
        f"{scores[other_rank]:>{column_width}}" for other_rank in teams
    ))

assert all(score_matrix[rank][rank] == 102 for rank in teams)
assert all(
    score_matrix[left][right] == score_matrix[right][left]
    for left in teams for right in teams
)
print(f"Validated a symmetric {len(teams)} × {len(teams)} score matrix.")

## Find score-linked groups

A pair is included only when its score equals the selected score. Groups are connected components: if A matches B and B matches C at that score, all three players are grouped together even if A and C are not directly paired.

## Exact-score pairs and player groups

Team comparison, pairwise matrix generation, exact-score pair lookup, and transitive player grouping are provided by the public `pyvgc` API. Grouping follows connected components: a chain of matching pairs joins its members even if not every pair within a group directly matches.

In [ ]:
target_score_pairs = 102

matching_pairs = find_team_pairs_by_score(score_matrix, target_score_pairs)
print(f"{len(matching_pairs)} distinct pairs score exactly {target_score_pairs}:")
for rank, other_rank, score in matching_pairs:
    print(
        f"{rank} ({teams[rank]['player']}) — "
        f"{other_rank} ({teams[other_rank]['player']}): {score}"
    )

score_groups = group_identical_teamplayers(score_matrix)
print("\nGroups of players with identical teams (score 102):")
for group in score_groups:
    members = ", ".join(
        f"{rank} ({teams[rank]['player']})" for rank in group
    )
    print(members)